In [1]:
pip install boto3

DEPRECATION: Loading egg at /opt/homebrew/lib/python3.12/site-packages/cvxopt-0.0.0-py3.12-macosx-14.0-arm64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import json
import csv
import os
from datetime import datetime

In [3]:
API_KEY = '8ic5iHAYy80bieTq4a9BILS8Y41SihodkpYOAeEF'  # Get from https://www.eia.gov/opendata/register.php
BASE_URL = 'https://api.eia.gov/v2/crude-oil-imports/data/'

In [4]:
MAX_SIZE_BYTES = 500 * 1024 * 1024  # 500 MB
output_folder = 'data_oil'
os.makedirs(output_folder, exist_ok=True)

In [5]:
params = {
    'api_key': API_KEY,
    'frequency': 'monthly',
    'data[0]': 'quantity',
    'sort[0][column]': 'period',
    'sort[0][direction]': 'desc',
    'offset': 0,
    'length': 5000
}

headers = {
    'X-Params': json.dumps({
        "frequency": "monthly",
        "data": ["quantity"],
        "facets": {},
        "start": "2025-01",
        "end": "2025-02",
        "sort": [{"column": "period", "direction": "desc"}],
        "offset": 0,
        "length": 5000
    })
}

all_records = []
offset = 0
total = None
estimated_size = 0

print('Fetching EIA crude oil import data...')

Fetching EIA crude oil import data...


In [ ]:
while True:
    params['offset'] = offset
    response = requests.get(BASE_URL, params=params, headers=headers)

    if response.status_code != 200:
        print(f'Error {response.status_code}: {response.text}')
        break

    data = response.json()
    records = data.get('response', {}).get('data', [])
    total = int(data.get('response', {}).get('total', 0))

    if not records:
        break

    # Estimate size of this batch in bytes
    batch_size = len(json.dumps(records).encode('utf-8'))

    # Stop if adding this batch would exceed 500 MB
    if estimated_size + batch_size > MAX_SIZE_BYTES:
        # Only add records up to the limit
        for record in records:
            record_size = len(json.dumps(record).encode('utf-8'))
            if estimated_size + record_size > MAX_SIZE_BYTES:
                break
            all_records.append(record)
            estimated_size += record_size
        print(f'500 MB limit reached. Stopping early.')
        break

    all_records.extend(records)
    estimated_size += batch_size

    pct = (len(all_records) / total * 100) if total else 0
    print(f'Fetched {len(all_records):,} / {total:,} records ({pct:.1f}%) | Size: {estimated_size / 1024 / 1024:.1f} MB')

    if len(all_records) >= total:
        break

    offset += 5000

# Save as JSON
json_file = os.path.join(output_folder, f'crude_oil_imports_{datetime.today().strftime("%Y%m%d")}.json')
with open(json_file, 'w') as f:
    json.dump(all_records, f, indent=2)

json_size = os.path.getsize(json_file) / 1024 / 1024
print(f'Saved JSON: {json_file} ({json_size:.1f} MB)')

# Save as CSV
if all_records:
    csv_file = os.path.join(output_folder, f'crude_oil_imports_{datetime.today().strftime("%Y%m%d")}.csv')
    with open(csv_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_records[0].keys())
        writer.writeheader()
        writer.writerows(all_records)

    csv_size = os.path.getsize(csv_file) / 1024 / 1024
    print(f'Saved CSV: {csv_file} ({csv_size:.1f} MB)')

print(f'Done. Total records: {len(all_records):,} | Folder: {output_folder}/')

Fetched 4,030 / 4,034 records (99.9%) | Size: 1.4 MB


In [ ]:
import boto3
from botocore.exceptions import NoCredentialsError
from config import AWS_ACCESS_KEY, AWS_SECRET_KEY, AWS_REGION, BUCKET_NAME

FILE_PATH = csv_file
S3_KEY = f"oil_data/{os.path.basename(csv_file)}"

s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

try:
    s3_client.upload_file(FILE_PATH, BUCKET_NAME, S3_KEY)
    print(f"Uploaded {FILE_PATH} to s3://{BUCKET_NAME}/{S3_KEY}")

except Exception as e:
    print("Upload failed:", str(e))
